In [156]:
#IMPORT LIBRARIES FOR NLP PROCESSING
import nltk                                     #Used for tokenization, stopwords, lemmatization
from nltk.tokenize import word_tokenize         #Split sentences into tokens (word-level)
from nltk.corpus import stopwords               #Stopword set to remove nonsense words (the, is, a... )          
from nltk.stem import WordNetLemmatizer         #Lemmatize
nltk.download('punkt', 'stopwords', 'wordnet')

#IMPORT SPACY FOR POS TAGGING + NER
import spacy                                    #Powerful NLP library for POS and NER
from spacy import displacy                      #spaCy's visualizer to highlight entities

#IMPORT OTHER UTILITIES
import re                                       #Regular expressions to clean text
import pandas as pd                             #Load and process CSV dataset

#IMPORT MATPLOTLIB FOR VISUALIZATION
import matplotlib.pyplot as plt                 #Used to draw graphs (POS, NER distribution)

In [157]:
# Load CSV file into a pandas DataFrame
tech_news_data = pd.read_csv("tech_startup_news_2025.csv")

In [158]:
tech_news_data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,30000,Amazon launches robotics system for vision model,"Mon, 27 Jan 2025 15:27:39 GMT",https://news.example.com/30000,https://news.example.com/30000,"Amazon reports new developments in energy, lev..."
1,1,30001,Amazon announces partnership with edge AI for ...,"Fri, 02 Aug 2024 02:35:55 GMT",https://news.example.com/30001,https://news.example.com/30001,"Amazon reports new developments in robotics, l..."
2,2,30002,Google unveils new vision model AI model,"Mon, 18 Nov 2024 11:01:11 GMT",https://news.example.com/30002,https://news.example.com/30002,"Google reports new developments in defense, le..."
3,3,30003,Anthropic releases multimodal model for LLM ap...,"Sun, 03 Nov 2024 10:50:49 GMT",https://news.example.com/30003,https://news.example.com/30003,"Anthropic reports new developments in finance,..."
4,4,30004,Salesforce announces partnership with MoE for ...,"Tue, 06 Aug 2024 09:34:59 GMT",https://news.example.com/30004,https://news.example.com/30004,"Salesforce reports new developments in energy,..."


In [159]:
tech_news_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [160]:
# Extract only the 'title' column and store it in a new DataFrame
titles = pd.DataFrame(tech_news_data['title'])
titles.head()

,title
0,Amazon launches robotics system for vision model
1,Amazon announces partnership with edge AI for ...
2,Google unveils new vision model AI model
3,Anthropic releases multimodal model for LLM ap...
4,Salesforce announces partnership with MoE for ...


In [161]:
#First, we convert the text to lowercase using .lower()
titles['lowercase'] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...
2,Google unveils new vision model AI model,google unveils new vision model ai model
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...


In [162]:
#Next, remove stopwords

en_stopwords = stopwords.words('english')
titles['no_stopwords'] = titles['lowercase'].apply(
    lambda x: ' '.join(word for word in x.split() if word not in en_stopwords))
titles.head()

,title,lowercase,no_stopwords
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model,amazon launches robotics system vision model
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...,amazon announces partnership edge ai salesforc...
2,Google unveils new vision model AI model,google unveils new vision model ai model,google unveils new vision model ai model
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...,anthropic releases multimodal model llm applic...
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...,salesforce announces partnership moe bytedance...


In [163]:
#Punctuation removal
titles['no_stopwords_no_punct'] = titles['no_stopwords'].str.replace(r"[^\w\s]"," ", regex=True)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model,amazon launches robotics system vision model,amazon launches robotics system vision model
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...,amazon announces partnership edge ai salesforc...,amazon announces partnership edge ai salesforc...
2,Google unveils new vision model AI model,google unveils new vision model ai model,google unveils new vision model ai model,google unveils new vision model ai model
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...,anthropic releases multimodal model llm applic...,anthropic releases multimodal model llm applic...
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...,salesforce announces partnership moe bytedance...,salesforce announces partnership moe bytedance...


In [164]:
#Tokenize
#Raw Tokens → from original title (keep everything)
titles['token_raw'] = titles['title'].map(word_tokenize)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct,token_raw
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model,amazon launches robotics system vision model,amazon launches robotics system vision model,"[Amazon, launches, robotics, system, for, visi..."
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...,amazon announces partnership edge ai salesforc...,amazon announces partnership edge ai salesforc...,"[Amazon, announces, partnership, with, edge, A..."
2,Google unveils new vision model AI model,google unveils new vision model ai model,google unveils new vision model ai model,google unveils new vision model ai model,"[Google, unveils, new, vision, model, AI, model]"
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...,anthropic releases multimodal model llm applic...,anthropic releases multimodal model llm applic...,"[Anthropic, releases, multimodal, model, for, ..."
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...,salesforce announces partnership moe bytedance...,salesforce announces partnership moe bytedance...,"[Salesforce, announces, partnership, with, MoE..."


In [165]:
#Cleaned Tokens → from preprocessed text (lowercase + no stopwords + no punct)
titles['tokens_clean'] = titles['no_stopwords_no_punct'].map(word_tokenize)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct,token_raw,tokens_clean
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model,amazon launches robotics system vision model,amazon launches robotics system vision model,"[Amazon, launches, robotics, system, for, visi...","[amazon, launches, robotics, system, vision, m..."
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...,amazon announces partnership edge ai salesforc...,amazon announces partnership edge ai salesforc...,"[Amazon, announces, partnership, with, edge, A...","[amazon, announces, partnership, edge, ai, sal..."
2,Google unveils new vision model AI model,google unveils new vision model ai model,google unveils new vision model ai model,google unveils new vision model ai model,"[Google, unveils, new, vision, model, AI, model]","[google, unveils, new, vision, model, ai, model]"
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...,anthropic releases multimodal model llm applic...,anthropic releases multimodal model llm applic...,"[Anthropic, releases, multimodal, model, for, ...","[anthropic, releases, multimodal, model, llm, ..."
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...,salesforce announces partnership moe bytedance...,salesforce announces partnership moe bytedance...,"[Salesforce, announces, partnership, with, MoE...","[salesforce, announces, partnership, moe, byte..."


In [166]:
#Lemmatizing
lemmatizer = WordNetLemmatizer()
titles['tokens_clean_lemmatized'] = titles['tokens_clean'].apply(
    lambda x: [lemmatizer.lemmatize(token) for token in x]
)
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punct,token_raw,tokens_clean,tokens_clean_lemmatized
0,Amazon launches robotics system for vision model,amazon launches robotics system for vision model,amazon launches robotics system vision model,amazon launches robotics system vision model,"[Amazon, launches, robotics, system, for, visi...","[amazon, launches, robotics, system, vision, m...","[amazon, launch, robotics, system, vision, model]"
1,Amazon announces partnership with edge AI for ...,amazon announces partnership with edge ai for ...,amazon announces partnership edge ai salesforc...,amazon announces partnership edge ai salesforc...,"[Amazon, announces, partnership, with, edge, A...","[amazon, announces, partnership, edge, ai, sal...","[amazon, announces, partnership, edge, ai, sal..."
2,Google unveils new vision model AI model,google unveils new vision model ai model,google unveils new vision model ai model,google unveils new vision model ai model,"[Google, unveils, new, vision, model, AI, model]","[google, unveils, new, vision, model, ai, model]","[google, unveils, new, vision, model, ai, model]"
3,Anthropic releases multimodal model for LLM ap...,anthropic releases multimodal model for llm ap...,anthropic releases multimodal model llm applic...,anthropic releases multimodal model llm applic...,"[Anthropic, releases, multimodal, model, for, ...","[anthropic, releases, multimodal, model, llm, ...","[anthropic, release, multimodal, model, llm, a..."
4,Salesforce announces partnership with MoE for ...,salesforce announces partnership with moe for ...,salesforce announces partnership moe bytedance...,salesforce announces partnership moe bytedance...,"[Salesforce, announces, partnership, with, MoE...","[salesforce, announces, partnership, moe, byte...","[salesforce, announces, partnership, moe, byte..."


In [167]:
#Flatten List of Lists into One Long List
#Approach 1
tokens_raw_list = sum(titles['token_raw'],[])
tokens_clean_list = sum(titles['tokens_clean_lemmatized'],[])
#print(...)

In [168]:
#Approach 2
tokens_raw_list = [token for tokens in titles['token_raw'] for token in tokens]
tokens_clean_list = [token for tokens in titles['tokens_clean_lemmatized'] for token in tokens]
#print(...)

In [169]:
#Approach 3
from itertools import chain
tokens_raw_list = list(chain.from_iterable(titles['token_raw']))
tokens_clean_list = list(chain.from_iterable(titles['tokens_clean_lemmatized']))
#print(...)

===================================================================================================

In [170]:
#POST TAGGING

In [171]:
nlp = spacy.load('en_core_web_sm')

In [172]:
spacy_doc = nlp(' '.join(tokens_raw_list))

In [173]:
#Convert spaCy's processed tokens into a pandas DataFrame for easy analysis
pos_df = pd.DataFrame({
    'token': [token.text for token in spacy_doc],    #Raw word/string
    'pos': [token.pos_ for token in spacy_doc],      #Universal POS (e.g., NOUN, VERB)
    'tag':[token.tag_ for token in spacy_doc]        #Detailed tag (e.g., NNP, VBZ)
}   
                   )
pos_df.head()

,token,pos,tag
0,Amazon,PROPN,NNP
1,launches,VERB,VBZ
2,robotics,NOUN,NNS
3,system,NOUN,NN
4,for,ADP,IN


In [174]:
#Count token frequency with detailed POS information
pos_df_counts = (
    pos_df
    .value_counts(['token', 'pos', 'tag'])            # Count identical (word + POS) combos
    .reset_index(name='counts')                       # Turn index → columns + name count column
    .sort_values(by="counts", ascending=False)        # Highest frequency first
)
pos_df_counts.head()

,token,pos,tag,counts
0,for,ADP,IN,578
1,AI,PROPN,NNP,408
2,model,NOUN,NN,383
3,in,ADP,IN,288
4,robotics,NOUN,NNS,162


In [175]:
# Get the 10 most frequent NOUNs (properly sorted)
top_nouns = pos_df_counts[pos_df_counts['pos'] == 'NOUN'].head(10)
top_nouns

,token,pos,tag,counts
2,model,NOUN,NN,383
4,robotics,NOUN,NNS,162
7,partnership,NOUN,NN,159
11,technology,NOUN,NN,151
10,initiative,NOUN,NN,151
9,invests,NOUN,NNS,151
15,system,NOUN,NN,142
18,m,NOUN,NN,139
19,applications,NOUN,NNS,138
22,datacenter,NOUN,NN,137


In [176]:
# Get the 10 most frequent VERBs (properly sorted)
top_verbs = pos_df_counts[pos_df_counts['pos'] == 'VERB'].head(10)
top_verbs

,token,pos,tag,counts
5,announces,VERB,VBZ,159
13,launches,VERB,VBZ,142
14,raises,VERB,VBZ,142
21,expands,VERB,VBZ,137
25,unveils,VERB,VBZ,131
29,releases,VERB,VBZ,125
39,rendering,VERB,VBG,83
70,platform,VERB,VBP,5
77,m,VERB,VBP,2


In [177]:
# Get the 10 most frequent ADJECTIVEs (properly sorted)
top_adj = pos_df_counts[pos_df_counts['pos'] == 'ADJ'].head(10)
top_adj

,token,pos,tag,counts
20,multimodal,ADJ,JJ,138
26,new,ADJ,JJ,131
36,neural,ADJ,JJ,85
58,quantum,ADJ,JJ,25
59,autonomous,ADJ,JJ,24
61,MoE,ADJ,JJ,20
66,Anthropic,ADJ,JJ,15
73,Salesforce,ADJ,JJ,2


===================================================================================================

In [178]:
#NER (Named Entity Recognition)

In [179]:
# Extract Named Entities (NER) – clean & modern way
ner_df = pd.DataFrame({
    'entity'  : [ent.text for ent in spacy_doc.ents],          #Entity's Name
    'ner_tag' : [ent.label_ for ent in spacy_doc.ents]         #the NER category (PERSON, ORG, DATE...)

}
                     )
ner_df.head(20)

,entity,ner_tag
0,Amazon,ORG
1,Amazon,ORG
2,AI for Salesforce technology Google,ORG
3,AI,GPE
4,Anthropic,NORP
5,LLM,ORG
6,MoE,ORG
7,ByteDance,ORG
8,AGI,ORG
9,AI,ORG


In [180]:
# Count entity frequency with detailed NER information
ner_df_counts = (
    ner_df
    .value_counts(['entity','ner_tag'])            # Count identical (entity + type) combos
    .reset_index(name='counts')                    # Convert MultiIndex → proper columns
    .sort_values(by="counts", ascending=False)     # Most frequent first
)
ner_df_counts

,entity,ner_tag,counts
0,AI,GPE,164
1,AGI,ORG,115
2,IBM,ORG,65
3,Microsoft,ORG,63
4,Amazon,ORG,59
...,...,...,...
190,2930,MONEY,1
191,2889,MONEY,1
192,2861,MONEY,1
193,2844,MONEY,1
